In [ ]:
# ==============================================================================
# SCRIPT: ANÁLISE INDIVIDUAL POR CÉLULA (MAPA + SÉRIES + STL)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose

# ==============================
# CONFIGURAÇÃO
# ==============================
TARGET_VAR = 'dV'   # 'dV' (Vertical) ou 'dH' (Horizontal)
GRID_SIZE = 50      # Aumente isto se tiver muitas células (ex: 100, 150)

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs
# ==============================
# Certifique-se que estes ficheiros correspondem à zona da barragem escolhida acima

# Alqueva
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

#Castelo do Bode
#asc_file = "data/castelo_bode_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0233_IW1_VV_2019_2023_1/EGMS_L2b_147_0233_IW1_VV_2019_2023_1.csv"
#desc_file = "data/castelo_bode_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0841_IW3_VV_2019_2023_1/EGMS_L2b_052_0841_IW3_VV_2019_2023_1.csv"

#norte_min, norte_max = 2017250, 2018550
#este_min, este_max = 2754250, 2754850

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Processamento (Melt, Interpolate, IDW, dV/dH)
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

def interpolate_ps(df, dates):
    if df.empty: return pd.DataFrame()
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

if asc_interp.empty:
    print("ERRO: Sem dados após interpolação. Verifique coordenadas.")
    exit()

def idw(source, target, radius=150, power=2):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=5, distance_upper_bound=radius)
        vals, thetas = [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)

asc_interp = idw(desc_interp, asc_interp).dropna(subset=['disp_idw'])

# Calcular Componente
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))
def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH
asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 3. Grelha e Agregação
# ==============================
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+GRID_SIZE, GRID_SIZE)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+GRID_SIZE, GRID_SIZE)
xs, ys = xe - GRID_SIZE/2, ye - GRID_SIZE/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Agregação
agg = asc_interp.groupby(['cell_id','date']).agg(val=(TARGET_VAR,'mean')).reset_index()
agg.rename(columns={'val': TARGET_VAR}, inplace=True) # Nome correto da coluna

# ==============================
# 4. Recorte
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
valid_cells = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid_cells.add(ids[idx])

# Grelha final
grid_sel = grid_recort[grid_recort['cell_id'].isin(valid_cells)].copy()

# ==============================
# 5. PREPARAÇÃO PARA PLOT (SEM CLUSTERING)
# ==============================
# Atribuir IDs numéricos simples (1, 2, 3...) para identificar nos gráficos
grid_sel = grid_sel.sort_values('cell_id').reset_index(drop=True)
grid_sel['simple_id'] = range(1, len(grid_sel) + 1)

# Mapear cell_id original para simple_id
id_map = dict(zip(grid_sel['cell_id'], grid_sel['simple_id']))
agg = agg[agg['cell_id'].isin(valid_cells)].copy()
agg['simple_id'] = agg['cell_id'].map(id_map)

# Pivot para ter matriz (Index=SimpleID, Cols=Dates)
agg_pivot = agg.pivot(index='simple_id', columns='date', values=TARGET_VAR)

# Cores: Uma cor distinta para cada célula (usar cmap 'tab20' ou similar)
n_cells = len(grid_sel)
cmap = plt.get_cmap('tab20') if n_cells <= 20 else plt.get_cmap('gist_ncar')
colors = {i: cmap(i/n_cells) for i in range(n_cells)}

print(f"Total de Células Analisadas: {n_cells}")

# ==============================
# FIGURA 1: MAPA IDENTIFICADO
# ==============================
print("Gerando Mapa...")
fig1, ax1 = plt.subplots(figsize=(10, 10))
grid.to_crs(epsg=3857).boundary.plot(ax=ax1, color='white', lw=0.3, alpha=0.3)
grid_sel.boundary.plot(ax=ax1, color='black', lw=1)

# Plot das células com número
grid_sel.plot(ax=ax1, facecolor='none', edgecolor='black', linewidth=1.5)
for idx, row in grid_sel.iterrows():
    # Calcular centróide para o texto
    cent = row.geometry.centroid
    ax1.text(cent.x, cent.y, str(row['simple_id']), fontsize=12, color='red', fontweight='bold', ha='center', va='center')

ctx.add_basemap(ax1, source=ctx.providers.Esri.WorldImagery)
ax1.set_axis_off()
ax1.set_title(f"Localização das Células ({n_cells})", fontsize=14)
plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (SUBPLOTS)
# ==============================
print("Gerando Séries...")
# Layout dinâmico: Tentar fazer uma grelha quadrada (ex: 4x4, 5x5)
cols = int(np.ceil(np.sqrt(n_cells)))
rows = int(np.ceil(n_cells / cols))

fig2, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows), sharex=True, sharey=True)
axes = axes.flatten() # Achatar para iterar fácil

ymin, ymax = agg[TARGET_VAR].min(), agg[TARGET_VAR].max()
pad = (ymax - ymin) * 0.1

for i in range(len(axes)):
    ax = axes[i]
    if i < n_cells:
        sid = i + 1 # Simple ID é 1-based
        data = agg_pivot.loc[sid]
        
        ax.plot(data.index, data.values, color='black', lw=1.0)
        ax.set_title(f"Célula {sid}", fontsize=10, fontweight='bold', color='red')
        ax.set_ylim(ymin - pad, ymax + pad)
        #ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        
        if i % cols == 0: ax.set_ylabel(f'{TARGET_VAR} (mm)')
    else:
        ax.axis('off') # Esconder subplots vazios

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: STL (COM ESCALA UNIFORME)
# ==============================
print("Gerando STL com escala uniforme...")

# 1. PRÉ-CÁLCULO: Calcular Decomposições e Limites Globais
decomps_storage = {}
vals_obs, vals_trend, vals_seas, vals_resid = [], [], [], []

for i in range(n_cells):
    sid = i + 1
    series = agg_pivot.loc[sid]
    try:
        # Calcular STL
        res = seasonal_decompose(pd.Series(series.values, index=series.index), 
                                 period=12, model='additive', extrapolate_trend='freq')
        decomps_storage[sid] = res
        
        # Guardar valores para cálculo de limites
        vals_obs.append(res.observed)
        vals_trend.append(res.trend)
        vals_seas.append(res.seasonal)
        vals_resid.append(res.resid)
    except:
        decomps_storage[sid] = None

# Função para calcular limites com margem de 10%
def get_global_limits(values_list):
    if not values_list: return (0, 1)
    # Concatenar todas as séries para achar min/max global
    all_vals = pd.concat(values_list)
    vmin, vmax = all_vals.min(), all_vals.max()
    margin = (vmax - vmin) * 0.1
    if margin == 0: margin = 0.1
    return (vmin - margin, vmax + margin)

# Calcular os limites globais
ylim_obs = get_global_limits(vals_obs)
ylim_trend = get_global_limits(vals_trend)
ylim_seas = get_global_limits(vals_seas)
ylim_resid = get_global_limits(vals_resid)

# 2. PLOTAGEM
fig3, axes = plt.subplots(n_cells, 4, figsize=(16, 1.0 * n_cells), sharex=True)
if n_cells == 1: axes = axes.reshape(1, 4)

for i in range(n_cells):
    sid = i + 1
    res = decomps_storage.get(sid)
    
    # Se a decomposição falhou, saltamos
    if res is None: continue

    # --- Observed ---
    ax = axes[i, 0]
    ax.plot(res.observed.index, res.observed, color='black', lw=1.0)
    ax.set_ylim(ylim_obs) # Escala Uniforme
    ax.set_ylabel(f'Célula {sid}', fontweight='bold', color='red', fontsize=9)
    if i == 0: ax.set_title("Observed (Original)")
    #ax.grid(True, linestyle=':', alpha=0.3)

    # --- Trend ---
    ax = axes[i, 1]
    ax.plot(res.trend.index, res.trend, color='blue', lw=1.0)
    ax.set_ylim(ylim_trend) # Escala Uniforme
    if i == 0: ax.set_title("Trend (Tendência)")
    #ax.grid(True, linestyle=':', alpha=0.3)

    # --- Seasonal ---
    ax = axes[i, 2]
    ax.plot(res.seasonal.index, res.seasonal, color='green', lw=1.0)
    ax.set_ylim(ylim_seas) # Escala Uniforme
    if i == 0: ax.set_title("Seasonal (Sazonalidade)")
    #ax.grid(True, linestyle=':', alpha=0.3)

    # --- Residual ---
    ax = axes[i, 3]
    ax.scatter(res.resid.index, res.resid, color='gray', s=5, alpha=0.7)
    ax.axhline(0, c='k', ls='--', lw=0.5)
    ax.set_ylim(ylim_resid) # Escala Uniforme
    if i == 0: ax.set_title("Residual (Ruído)")
    #ax.grid(True, linestyle=':', alpha=0.3)

    # Formatação X apenas na última linha
    if i == n_cells - 1:
        for ax_col in axes[i, :]: 
            #ax_col.xaxis.set_major_formatter(mdates.DateFormatter("'%y"))
            ax_col.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

# Nivelamento

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import math

# ==========================================
# 1. CONFIGURAÇÃO
# ==========================================
file_path = "data/nivelamento.xlsx"

# Lista dos instrumentos que quer analisar
instrumentos_alvo = ['N3', 'N5', 'N6', 'N7', 'N8', 'N9', 'N10']

# ==========================================
# 2. CARREGAMENTO E FILTRO
# ==========================================
try:
    df = pd.read_excel(file_path)
    # Garantir formato de data
    df['data'] = pd.to_datetime(df['data'])
    
    # Filtrar apenas os instrumentos desejados
    df = df[df['instrumento'].isin(instrumentos_alvo)]
    
    if df.empty:
        print("Aviso: Nenhum dado encontrado para os instrumentos selecionados.")
        exit()
        
except FileNotFoundError:
    print(f"Erro: O ficheiro '{file_path}' não foi encontrado.")
    exit()

df['deslocamento (m)'] = pd.to_numeric(
    df['deslocamento (m)'].astype(str).str.replace(',', '.', regex=False),
    errors='coerce'
)

# ==========================================
# 3. PLOTAGEM (SUBPLOTS)
# ==========================================
# Calcular layout da grelha (ex: 2 colunas)
n_inst = len(instrumentos_alvo)
cols = 2
rows = math.ceil(n_inst / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows), constrained_layout=True)
axes = axes.flatten() # Achatar para facilitar o loop

for i, inst in enumerate(instrumentos_alvo):
    ax = axes[i]
    
    # Dados deste instrumento
    subset = df[df['instrumento'] == inst].sort_values('data')
    
    if not subset.empty:
        ax.plot(subset['data'], subset['deslocamento (m)'], 
                marker='o', markersize=4, linestyle='-', linewidth=1.5, color='tab:blue')
        
        # Formatação individual
        ax.set_title(f"Instrumento: {inst}", fontweight='bold', color='navy')
        ax.set_ylabel("Deslocamento (m)")
        ax.grid(True, linestyle=':', alpha=0.7)
        
        # Formatação Datas
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.tick_params(axis='x', rotation=45)
    else:
        ax.text(0.5, 0.5, 'Sem dados', ha='center', va='center', transform=ax.transAxes)

# Esconder eixos vazios se houver (caso ímpar)
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle(f"Monitorização de Nivelamento ({len(instrumentos_alvo)} Instrumentos)", fontsize=16, y=1.02)
plt.show()

# Comparação EGMS com Nivelamento geodesia

In [ ]:
# ==============================================================================
# COMPARAÇÃO FINAL: CÉLULA InSAR vs NIVELAMENTO (TENDÊNCIAS CONTÍNUAS)
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# --- CONFIGURAÇÃO ---
CELULA_ID_ALVO = 18       
INSTRUMENTO_ALVO = 'N7'   
NIVELAMENTO_PATH = "data/nivelamento.xlsx"
DATA_INICIO = '2019-01-01'
GRAU_TENDENCIA = 3  # 1 = Reta, 2 = Curva

print(f"\n{'='*60}")
print(f" A COMPARAR: Célula {CELULA_ID_ALVO} vs {INSTRUMENTO_ALVO}")
print(f"{'='*60}")

# 1. Obter InSAR
ts_insar = None
try:
    if 'agg_pivot' in locals():
        ts_insar = agg_pivot.loc[CELULA_ID_ALVO]
        ts_insar = ts_insar[ts_insar.index >= pd.to_datetime(DATA_INICIO)]
        if ts_insar.empty: print("AVISO: InSAR vazio."); ts_insar = None
except KeyError: print("ERRO: Célula não encontrada.")

# 2. Obter Nivelamento
if ts_insar is not None:
    try:
        df_lev = pd.read_excel(NIVELAMENTO_PATH)
        df_lev['data'] = pd.to_datetime(df_lev['data'])
        
        # <<< AQUI >>>
        df_lev['deslocamento (m)'] = pd.to_numeric(
            df_lev['deslocamento (m)']
            .astype(str)
            .str.replace(',', '.', regex=False),
            errors='coerce'
        )
        
        df_inst = df_lev[
            (df_lev['instrumento'] == INSTRUMENTO_ALVO) & 
            (df_lev['data'] >= pd.to_datetime(DATA_INICIO))
        ].sort_values('data').dropna(subset=['deslocamento (m)']).copy()
        
        if not df_inst.empty:
            # Converter m -> mm
            col_valor = 'deslocamento (m)' # Ajuste se necessário
            if col_valor not in df_inst.columns:
                 # Procura automática
                 cands = [c for c in df_inst.columns if 'deslocamento' in c.lower()]
                 if cands: col_valor = cands[0]
            
            #df_inst['disp_mm'] = df_inst[col_valor]
            df_inst['disp_mm'] = df_inst[col_valor] * 1000  # Converter para mm

            # Normalizar (Centrar ambas no zero para comparar evolução)
            insar_norm = ts_insar - ts_insar.mean()
            nivel_norm = df_inst['disp_mm'] - df_inst['disp_mm'].mean()

            # --- NOVA FUNÇÃO DE TENDÊNCIA (GERA CURVA CONTÍNUA) ---
            def get_trend_curve(dates_input, values_input, degree, date_range_full):
                # 1. Ajustar o modelo aos dados REAIS
                x_input = mdates.date2num(dates_input)
                
                # Proteção contra poucos dados
                if len(x_input) <= degree:
                    return np.full(len(date_range_full), np.nan), " (Dados insuf.)"

                coeffs = np.polyfit(x_input, values_input, degree)
                poly_func = np.poly1d(coeffs)
                
                # 2. Gerar a linha para TODO o intervalo de tempo (suave)
                x_full = mdates.date2num(date_range_full)
                y_trend = poly_func(x_full)
                
                # Texto da legenda
                lbl = ""
                if degree == 1:
                    vel = coeffs[0] * 365.25
                    lbl = f" ({vel:.1f} mm/ano)"
                
                return y_trend, lbl

            # Criar um eixo de tempo contínuo comum para as linhas de tendência ficarem bonitas
            datas_comuns_plot = pd.date_range(start=ts_insar.index.min(), end=ts_insar.index.max(), freq='D')

            # Calcular Tendências (usando o eixo contínuo para o resultado)
            trend_insar, lbl_i = get_trend_curve(ts_insar.index, insar_norm.values, GRAU_TENDENCIA, datas_comuns_plot)
            trend_nivel, lbl_n = get_trend_curve(df_inst['data'], nivel_norm.values, GRAU_TENDENCIA, datas_comuns_plot)

            # ==============================
            # PLOT
            # ==============================
            fig, ax = plt.subplots(figsize=(12, 5))

            # --- INSAR ---
            # Dados (Cinza/Preto)
            ax.plot(ts_insar.index, insar_norm, 
                    color='gray', alpha=0.4, lw=1, 
                    label=f'InSAR Célula {CELULA_ID_ALVO} (Dados)') # <--- CORRIGIDO

            # Tendência (Preto Forte)
            ax.plot(datas_comuns_plot, trend_insar, 
                    color='black', lw=2.5, ls='-', 
                    label=f'Tendência InSAR{lbl_i}') # <--- CORRIGIDO

            # --- NIVELAMENTO ---
            # Dados (Bolinhas Vermelhas soltas)
            ax.plot(df_inst['data'], nivel_norm, 
                    color='red', marker='o', linestyle='None', ms=6, alpha=0.6, 
                    label=f'Nivelamento {INSTRUMENTO_ALVO} (Pontos)') # <--- CORRIGIDO

            # Tendência (Vermelho Tracejado)
            ax.plot(datas_comuns_plot, trend_nivel, 
                    color='red', lw=2.5, ls='--', 
                    label=f'Tendência Nivel.{lbl_n}') # <--- CORRIGIDO

            ax.set_title(f"Comparação de Tendências: InSAR vs Topografia ({TARGET_VAR})", fontsize=14, fontweight='bold')
            ax.set_ylabel("Variação Relativa (mm)")
            ax.grid(True, ls=':', alpha=0.6)
            ax.legend()
            
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
            plt.xticks(rotation=0)
            
            plt.tight_layout()
            plt.show()

    except Exception as e:
        print(f"Erro na comparação: {e}")

In [ ]:
# ==============================================================================
# COMPARAÇÃO FINAL: CÉLULA InSAR vs NIVELAMENTO (TENDÊNCIAS CONTÍNUAS)
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# --- CONFIGURAÇÃO ---
CELULA_ID_ALVO = 18
INSTRUMENTO_ALVO = 'N7'
NIVELAMENTO_PATH = "data/nivelamento.xlsx"
DATA_INICIO = '2019-01-01'
GRAU_TENDENCIA = 3        # 1 = velocidade, 2 = aceleração
INSAR_EM_MM = True        # <<< MUITO IMPORTANTE

print(f"\n{'='*60}")
print(f" A COMPARAR: Célula {CELULA_ID_ALVO} vs {INSTRUMENTO_ALVO}")
print(f"{'='*60}")

# ==============================================================================
# 1. OBTÉM INSaR
# ==============================================================================
ts_insar = None

try:
    if 'agg_pivot' in locals():
        ts_insar = agg_pivot.loc[CELULA_ID_ALVO]
        ts_insar = ts_insar[ts_insar.index >= pd.to_datetime(DATA_INICIO)]

        if ts_insar.empty:
            print("AVISO: Série InSAR vazia.")
            ts_insar = None

        # Garantir unidade do InSAR
        if not INSAR_EM_MM:
            ts_insar = ts_insar * 1000


except KeyError:
    print("ERRO: Célula InSAR não encontrada.")
    ts_insar = None

# ==============================================================================
# 2. OBTÉM NIVELAMENTO
# ==============================================================================
if ts_insar is not None:

    try:
        df_lev = pd.read_excel(NIVELAMENTO_PATH)

        # Datas
        df_lev['data'] = pd.to_datetime(df_lev['data'])

        # Converter vírgula decimal e garantir float
        df_lev['deslocamento (m)'] = pd.to_numeric(
            df_lev['deslocamento (m)']
            .astype(str)
            .str.replace(',', '.', regex=False),
            errors='coerce'
        )

        # Filtrar instrumento
        df_inst = df_lev[
            (df_lev['instrumento'] == INSTRUMENTO_ALVO) &
            (df_lev['data'] >= pd.to_datetime(DATA_INICIO))
        ].sort_values('data').dropna(subset=['deslocamento (m)']).copy()

        if df_inst.empty:
            print("AVISO: Nivelamento vazio.")
            raise SystemExit

        # Converter metros → milímetros
        df_inst['disp_mm'] = df_inst['deslocamento (m)'] * 1000.0

        # Normalizar (centrar no zero)
        insar_norm = ts_insar - ts_insar.mean()
        nivel_norm = df_inst['disp_mm'] - df_inst['disp_mm'].mean()

        # ==============================================================================
        # 3. FUNÇÃO DE TENDÊNCIA (ESTÁVEL)
        # ==============================================================================
        def get_trend_curve(dates_input, values_input, degree, date_range_full):

            x = mdates.date2num(dates_input)
            x0 = x.mean()
            x = x - x0

            if len(x) <= degree:
                return np.full(len(date_range_full), np.nan), " (dados insuf.)"

            coeffs = np.polyfit(x, values_input, degree)
            poly = np.poly1d(coeffs)

            x_full = mdates.date2num(date_range_full) - x0
            y_trend = poly(x_full)

            lbl = ""
            if degree == 1:
                vel = coeffs[0] * 365.25
                lbl = f" ({vel:.1f} mm/ano)"

            return y_trend, lbl

        # Eixo temporal contínuo comum
        datas_comuns = pd.date_range(
            start=ts_insar.index.min(),
            end=ts_insar.index.max(),
            freq='D'
        )

        # Tendências
        trend_insar, lbl_i = get_trend_curve(
            ts_insar.index, insar_norm.values, GRAU_TENDENCIA, datas_comuns
        )

        trend_nivel, lbl_n = get_trend_curve(
            df_inst['data'], nivel_norm.values, GRAU_TENDENCIA, datas_comuns
        )

        # ==============================================================================
        # 4. PLOT
        # ==============================================================================
        fig, ax = plt.subplots(figsize=(12, 5))

        # InSAR
        ax.plot(ts_insar.index, insar_norm,
                color='gray', alpha=0.4, lw=1,
                label=f'InSAR Célula {CELULA_ID_ALVO}')

        ax.plot(datas_comuns, trend_insar,
                color='black', lw=2.5,
                label=f'Tendência InSAR{lbl_i}')

        # Nivelamento
        ax.plot(df_inst['data'], nivel_norm,
                color='red', marker='o', ls='None',
                ms=6, alpha=0.6,
                label=f'Nivelamento {INSTRUMENTO_ALVO}')

        ax.plot(datas_comuns, trend_nivel,
                color='red', lw=2.5, ls='--',
                label=f'Tendência Nivel.{lbl_n}')

        # Formatação
        ax.set_title(f"Comparação InSAR vs Nivelamento", fontsize=14, fontweight='bold')
        ax.set_ylabel("Variação Relativa (mm)")
        ax.grid(True, ls=':', alpha=0.6)
        ax.legend()
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Erro na comparação: {e}")


In [ ]:
# ==============================================================================
# COMPARAÇÃO MÚLTIPLA: PARES CÉLULA InSAR vs NIVELAMENTO
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# --- CONFIGURAÇÃO ---
NIVELAMENTO_PATH = "data/nivelamento.xlsx"
DATA_INICIO = '2019-01-01'
GRAU_TENDENCIA = 2  # 1=Reta, 2=Curva

# LISTA DE PARES A COMPARAR
# Formato: (ID_CELULA, 'NOME_INSTRUMENTO')
# Nota: Use IDs inteiros se o seu mapa tiver IDs inteiros (1, 2...), ou strings se forem strings ('50_50').

# Pares para a grelha de 50 m x 50 m
PARES_COMPARACAO = [
    (49, 'N1'),
    (50, 'N3'),
    (60, 'N3/N4'),
    (2, 'N5'), # ou o 3 
    (14, 'N6'), # ou o 14
    (18, 'N7'),
    (23, 'N8'),
    (30, 'N9'),
    (35, 'N10'),
]

print(f"\n{'='*60}")
print(f" INICIANDO COMPARAÇÃO MÚLTIPLA ({len(PARES_COMPARACAO)} Pares)")
print(f"{'='*60}")

# Função para calcular curva de tendência (igual à anterior)
def get_trend_curve(dates_input, values_input, degree, date_range_full):
    x_input = mdates.date2num(dates_input)
    if len(x_input) <= degree: return np.full(len(date_range_full), np.nan), " (Dados insuf.)"
    coeffs = np.polyfit(x_input, values_input, degree)
    poly_func = np.poly1d(coeffs)
    x_full = mdates.date2num(date_range_full)
    y_trend = poly_func(x_full)
    lbl = ""
    if degree == 1:
        vel = coeffs[0] * 365.25
        lbl = f" ({vel:.1f} mm/ano)"
    return y_trend, lbl

# Função para gerar um gráfico
def plot_comparison(celula_id, inst_nome, df_nivelamento):
    print(f"\n--- Processando Par: Célula {celula_id} vs {inst_nome} ---")
    
    # 1. Obter InSAR
    try:
        ts_insar = agg_pivot.loc[celula_id]
        ts_insar = ts_insar[ts_insar.index >= pd.to_datetime(DATA_INICIO)]
        if ts_insar.empty:
            print(f" -> AVISO: Célula {celula_id} sem dados > {DATA_INICIO}. Saltando.")
            return
    except KeyError:
        print(f" -> ERRO: Célula {celula_id} não existe. Saltando.")
        return

    # 2. Obter Nivelamento
    df_inst = df_nivelamento[
        (df_nivelamento['instrumento'] == inst_nome) & 
        (df_nivelamento['data'] >= pd.to_datetime(DATA_INICIO))
    ].sort_values('data').dropna(subset=['deslocamento (m)']).copy()

    if df_inst.empty:
        print(f" -> AVISO: Instrumento {inst_nome} sem dados > {DATA_INICIO}. Saltando.")
        return

    # 3. Converter e Normalizar
    col_valor = 'deslocamento (m)'
    if col_valor not in df_inst.columns:
         cands = [c for c in df_inst.columns if 'deslocamento' in c.lower()]
         if cands: col_valor = cands[0]
    
    df_inst['disp_mm'] = df_inst[col_valor]
    insar_norm = ts_insar - ts_insar.mean()
    nivel_norm = df_inst['disp_mm'] - df_inst['disp_mm'].mean()

    # 4. Tendências
    datas_comuns_plot = pd.date_range(start=ts_insar.index.min(), end=ts_insar.index.max(), freq='D')
    trend_insar, lbl_i = get_trend_curve(ts_insar.index, insar_norm.values, GRAU_TENDENCIA, datas_comuns_plot)
    trend_nivel, lbl_n = get_trend_curve(df_inst['data'], nivel_norm.values, GRAU_TENDENCIA, datas_comuns_plot)

    # 5. Plot
    fig, ax = plt.subplots(figsize=(10, 4))
    
    # InSAR
    ax.plot(ts_insar.index, insar_norm, color='gray', alpha=0.4, lw=1, 
            label=f'InSAR Célula {celula_id} (Dados)')
    ax.plot(datas_comuns_plot, trend_insar, color='black', lw=2.0, ls='-', 
            label=f'Tendência InSAR{lbl_i}')

    # Nivelamento
    ax.plot(df_inst['data'], nivel_norm, color='red', marker='o', ls='None', ms=6, alpha=0.6, 
            label=f'Nivelamento {inst_nome} (Pontos)')
    ax.plot(datas_comuns_plot, trend_nivel, color='red', lw=2.0, ls='--', 
            label=f'Tendência Nivel.{lbl_n}')

    # Info Box
    info_txt = f"Pontos InSAR: {len(ts_insar)}\nPontos Nivel.: {len(df_inst)}"
    ax.text(0.02, 0.05, info_txt, transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax.set_title(f"Célula {celula_id} vs Taco {inst_nome}", fontsize=12)
    ax.set_ylabel("Variação (mm)")
    #ax.grid(True, ls=':', alpha=0.6)
    ax.legend(loc='best', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    
    plt.tight_layout()
    plt.show()

# --- EXECUÇÃO DO LOOP ---
try:
    # Carregar Excel uma vez para eficiência
    df_lev_all = pd.read_excel(NIVELAMENTO_PATH)
    df_lev_all['data'] = pd.to_datetime(df_lev_all['data'])
    
    # <<< AQUI >>>
    df_lev_all['deslocamento (m)'] = pd.to_numeric(
        df_lev_all['deslocamento (m)']
        .astype(str)
        .str.replace(',', '.', regex=False),
        errors='coerce'
    )
    
    # Iterar sobre os pares
    for cel_id, inst_name in PARES_COMPARACAO:
        plot_comparison(cel_id, inst_name, df_lev_all)

except Exception as e:
    print(f"ERRO GERAL: {e}")

In [ ]:
# ==============================================================================
# COMPARAÇÃO MÚLTIPLA EM GRELHA: CÉLULA InSAR vs NIVELAMENTO
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import math

# --- CONFIGURAÇÃO ---
NIVELAMENTO_PATH = "data/nivelamento.xlsx"
DATA_INICIO = '2019-01-01'
GRAU_TENDENCIA = 3  # 1=Reta, 2=Curva

# CONFIGURAÇÃO DO LAYOUT
N_COLS = 3  # Número de colunas desejado
# O número de linhas será calculado automaticamente

# LISTA DE PARES A COMPARAR
PARES_COMPARACAO = [
    (49, 'N1'),
    (50, 'N3'),
    (60, 'N4'), # Exemplo: N3/N4 simplificado para N4 se for esse o nome exato
    (2,  'N5'), 
    (14, 'N6'), 
    (18, 'N7'),
    (23, 'N8'),
    (30, 'N9'),
    (35, 'N10'),
]

print(f"\n{'='*60}")
print(f" INICIANDO COMPARAÇÃO MÚLTIPLA ({len(PARES_COMPARACAO)} Pares)")
print(f"{'='*60}")

# Função para calcular curva de tendência (Reutilizável)
def get_trend_curve(dates_input, values_input, degree, date_range_full):
    x_input = mdates.date2num(dates_input)
    if len(x_input) <= degree: return np.full(len(date_range_full), np.nan), ""
    coeffs = np.polyfit(x_input, values_input, degree)
    poly_func = np.poly1d(coeffs)
    x_full = mdates.date2num(date_range_full)
    y_trend = poly_func(x_full)
    lbl = ""
    if degree == 1:
        vel = coeffs[0] * 365.25
        lbl = f" ({vel:.1f} mm/ano)"
    return y_trend, lbl

try:
    # 1. Carregar Excel
    df_lev_all = pd.read_excel(NIVELAMENTO_PATH)

    # Datas
    df_lev_all['data'] = pd.to_datetime(df_lev_all['data'])

    # <<< AQUI >>>
    df_lev_all['deslocamento (m)'] = pd.to_numeric(
        df_lev_all['deslocamento (m)']
        .astype(str)
        .str.replace(',', '.', regex=False),
        errors='coerce'
    )


    # 2. Configurar Subplots
    n_plots = len(PARES_COMPARACAO)
    n_rows = math.ceil(n_plots / N_COLS)
    
    # Tamanho dinâmico da figura (ajuste o 4 e o 15 conforme necessário)
    fig, axes = plt.subplots(n_rows, N_COLS, figsize=(5 * N_COLS, 4 * n_rows), constrained_layout=True)
    
    # Garantir que axes é sempre uma lista plana, mesmo que seja só 1 plot
    if n_plots == 1: axes = [axes]
    else: axes = axes.flatten()

    # 3. Loop de Plotagem
    for i, (cel_id, inst_nome) in enumerate(PARES_COMPARACAO):
        ax = axes[i]
        print(f"Processando: Célula {cel_id} vs {inst_nome}")

        # --- A. Obter InSAR ---
        ts_insar = None
        try:
            ts_insar = agg_pivot.loc[cel_id]
            ts_insar = ts_insar[ts_insar.index >= pd.to_datetime(DATA_INICIO)]
        except KeyError: pass

        # --- B. Obter Nivelamento ---
        df_inst = df_lev_all[
            (df_lev_all['instrumento'] == inst_nome) & 
            (df_lev_all['data'] >= pd.to_datetime(DATA_INICIO))
        ].sort_values('data').dropna(subset=['deslocamento (m)']).copy()

        # --- C. Validar e Plotar ---
        if ts_insar is not None and not ts_insar.empty and not df_inst.empty:
            
            # Normalizar
            col_valor = 'deslocamento (m)' # Ajuste se necessário
            if col_valor not in df_inst.columns:
                 cands = [c for c in df_inst.columns if 'deslocamento' in c.lower()]
                 if cands: col_valor = cands[0]
            
            df_inst['disp_mm'] = df_inst[col_valor]
            insar_norm = ts_insar - ts_insar.mean()
            nivel_norm = df_inst['disp_mm'] - df_inst['disp_mm'].mean()

            # Tendências
            datas_comuns = pd.date_range(start=ts_insar.index.min(), end=ts_insar.index.max(), freq='D')
            t_insar, l_i = get_trend_curve(ts_insar.index, insar_norm.values, GRAU_TENDENCIA, datas_comuns)
            t_nivel, l_n = get_trend_curve(df_inst['data'], nivel_norm.values, GRAU_TENDENCIA, datas_comuns)

            # Desenhar
            ax.plot(ts_insar.index, insar_norm, color='gray', alpha=0.4, lw=1)
            ax.plot(datas_comuns, t_insar, color='black', lw=1.5, ls='-', label=f'InSAR{l_i}')
            
            ax.plot(df_inst['data'], nivel_norm, color='red', marker='.', ls='None', ms=5, alpha=0.6)
            ax.plot(datas_comuns, t_nivel, color='red', lw=1.5, ls='--', label=f'Nivel.{l_n}')
            
            # Formatação
            ax.set_title(f"Célula {cel_id} vs {inst_nome}", fontsize=10, fontweight='bold')
            ax.legend(fontsize=8, loc='best')
            ax.grid(True, ls=':', alpha=0.5)
            
            # Formatar Datas apenas na última linha (opcional, aqui formato em todos para clareza)
            ax.xaxis.set_major_formatter(mdates.DateFormatter("'%y"))
            
        else:
            # Caso falhe (sem dados), escrever mensagem no plot vazio
            ax.text(0.5, 0.5, "Sem Dados", ha='center', va='center')
            ax.set_title(f"Célula {cel_id} vs {inst_nome}")

    # 4. Limpar eixos vazios (se N_PLOTS não for múltiplo de N_COLS)
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f"Comparação InSAR vs Nivelamento (Grau {GRAU_TENDENCIA})", fontsize=14, y=1.02)
    plt.show()

except Exception as e:
    print(f"ERRO GERAL: {e}")